# EyePrep
#### Version: 0.0.1
#### By: Martin SZINTE
#### With: Sina M. KLING
#### Description:
This code is a first attempt of a tool meant to check the quality and <br>
extract the main feature of eye-tracking data. It follows the agreement <br>
of data structure for eye-tracking made by the latest [BEP020](https://bids-specification--1128.org.readthedocs.build/en/1128/modality-specific-files/physiological-recordings.html#eye-tracking).<br>

## Plots suggestions
1. Main quality check (missing values, dvar?)
2. Timeseries with X/Y together with 2D plots with colorscale for time and lateral histograms
3. Classification (blink/saccade/pursuit/fixation) using https://github.com/MikhailStartsev/deep_em_classifier?tab=readme-ov-file
4. Saccade analysis (main sequence, saccade, microsaccade rate)
5. Pupil analysis / Blink analysis / 

## TODO:
- [x] Copy sourcedata from previous project
- [ ] Run eye2bids to get bids data
- [x] Make lists of metadata
- [ ] Preprocess data
  - [x] Load the experimental data
  - [x] Deal with settings of EyePrep
  - [x] Remove blinks
  - [x] Remove nans
  - [x] Check where is info for dva conversion
  - [ ] Convert to dva and change axis to center data
  - [ ] Normalise pupil data
  - [ ] Detrending
  - [ ] Downsampling
  - [ ] Smoothing
  - [ ] Save the preprocessed data
- [ ] Make plots
  - [ ] 
- [ ] Put them in a html file

In [ ]:
# Inputs
subjects = ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05',
            'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-11',
            'sub-12', 'sub-13', 'sub-14', 'sub-17', 'sub-20',
            'sub-21', 'sub-22', 'sub-23', 'sub-24', 'sub-25']

sessions = ['ses-01', 'ses-02', 'ses-02', 'ses-02', 'ses-02',
            'ses-02', 'ses-02', 'ses-02', 'ses-02', 'ses-02',
            'ses-02', 'ses-02', 'ses-02', 'ses-02', 'ses-02',
            'ses-02', 'ses-02', 'ses-02', 'ses-02', 'ses-02']

## Copy sourcedata

In [ ]:
# Imports
import os
import glob

# Set the source and destination directories
src_bids_dir = '/home/mszinte/disks/meso_S/data/RetinoMaps/sourcedata'
dst_bids_dir = '/home/mszinte/disks/meso_S/data/EyePrep/sourcedata'

for subject, session in zip(subjects, sessions):
    src_folder = f'{src_bids_dir}/{subject}/{session}/func/'
    dst_folder = f'{dst_bids_dir}/{subject}/{session}/func/'
    
    # Create new folder
    os.makedirs(dst_folder, exist_ok=True)

    # Get eyetracking files
    eye_files = glob.glob(os.path.join(src_folder, '*task-pMF*'))
    for eye_file in eye_files:
        dst_file = os.path.join(dst_folder, os.path.basename(eye_file))
        print(f'rsync -avuz {eye_file} {dst_file}')
        os.system(f'rsync -avuz {eye_file} {dst_file}')

    # Copy event files
    event_files = glob.glob(os.path.join(src_folder, '*task-pMF*events.tsv'))
    for event_file in event_files:
        dst_file = os.path.join(dst_folder, os.path.basename(event_file))
        os.system(f'rsync -avuz {event_file} {dst_file}')

## copy manually the dataset_description.json / participants.json / participant.tsv / task-pMF_events.json / README
## manually edit dataset_description.json and README

## Copy event file

In [ ]:
# Set the source and destination directories
src_bids_dir = '/home/mszinte/disks/meso_S/data/RetinoMaps'
dst_bids_dir = '/home/mszinte/disks/meso_S/data/EyePrep'

for subject, session in zip(subjects, sessions):
    src_folder = f'{src_bids_dir}/{subject}/{session}/func/'
    dst_folder = f'{dst_bids_dir}/{subject}/{session}/func/'
    
    # Create new folder
    os.makedirs(dst_folder, exist_ok=True)

    # Copy event files
    event_files = glob.glob(os.path.join(src_folder, '*task-pMF*events.tsv'))
    for event_file in event_files:
        dst_file = os.path.join(dst_folder, os.path.basename(event_file))
        os.system(f'rsync -avuz {event_file} {dst_file}')

## Convert sourcedata using eye2bids

## Define EyePrep settings
- `bids_dir`: main BIDS folder
- `eyeprep_dir`: EyePrep output folder
- `x_coord_column`: gaze X coordinate column names in dataset
- `y_coord_column`: gaze Y coordinate column names in dataset
- `pupil_column`: pupil coordinate column names in dataset
- `blink_removal`: Blink removal method ('pupil_off')
- `blink_buffer`: Period before and after detected blink to nan (in seconds) 

In [2]:
# Load inputs and settings
bids_dir = '/home/mszinte/disks/meso_S/data/EyePrep'
eyeprep_dir = f'{bids_dir}/derivatives/EyePrep/'
x_coord_column = 'x_coordinate'
y_coord_column = 'y_coordinate'
pupil_column = 'pupil_size'
blink_removal_method = 'pupil_off'
blink_buffer = 0.150

## Get lists of metadata
Subjects: `subjects`
Sessions for a specific subject: `sessions['sub-01']`<br>
Unique tasks found in the dataset: `tasks`<br>
Metadata found for specific task `events['task-pMF']`<br>
Unique modality found in the dataset: `modalities`<br>
Unique recordings found in the dataset: `recordings`<br>
Runs for a specific subject, session, modality, task, and recording: `runs['sub-01']['ses-01']['func']['task-pMF']['recording-eye1']`<br>
Metadata for a specific subject, session, modality, task, recording, and run: `metadata['sub-01']['ses-01']['func']['task-pMF']['recording-eye1']['run-01']`<br>
Filenames for a specific subject, session, modality, task, and recording: `filenames['sub-01']['ses-01']['func']['task-pMF']['recording-eye1']['run-01']['physioevents.json']`<br>

In [3]:
import os
import json

# Get the list of subjects
subjects = sorted([f for f in os.listdir(bids_dir) if f.startswith('sub')])

# Get the list of session folders for each subject
sessions = {}
for subject in subjects:
    subject_dir = os.path.join(bids_dir, subject)
    session_folders = sorted([d for d in os.listdir(subject_dir) if os.path.isdir(os.path.join(subject_dir, d))])
    sessions[subject] = session_folders

# Collect all unique modalities from folders containing _physio.tsv.gz
modalities = sorted({
    modality
    for subject in subjects
    for session in sessions[subject]
    for modality in os.listdir(os.path.join(bids_dir, subject, session))
    if os.path.isdir(os.path.join(bids_dir, subject, session, modality))
    and any(f.endswith('_physio.tsv.gz') for f in os.listdir(os.path.join(bids_dir, subject, session, modality)))
})

# Collect all unique task names
tasks = list(set(
    part
    for subject in subjects
    for session_dir in sessions[subject]
    for modality in modalities
    for f in os.listdir(os.path.join(bids_dir, subject, session_dir, modality))
    if f.endswith('_events.tsv')
    for part in f.split('_') if part.startswith('task-')
))

# Get task _events.json files
events = {}
for f in os.listdir(bids_dir):
    for task in tasks:
        if f.endswith('_events.json') and f.startswith('task-'):
            with open(os.path.join(bids_dir, f), 'r') as file:
                events[task] = json.load(file)

# Collect all unique recording types
recordings = list(set(
    part
    for subject in subjects
    for session_dir in sessions[subject]
    for modality in modalities
    for f in os.listdir(os.path.join(bids_dir, subject, session_dir, modality))
    if f.endswith('_physioevents.json')
    for part in f.split('_') if part.startswith('recording-')
))

# Get run list
runs = {}
for subject in subjects:
    runs[subject] = {}
    for session_dir in sessions[subject]:
        runs[subject][session_dir] = {}
        for modality in modalities:
            modality_path = os.path.join(bids_dir, subject, session_dir, modality)
            if not os.path.isdir(modality_path): continue
            runs[subject][session_dir][modality] = {}
            for task in tasks:
                runs[subject][session_dir][modality][task] = {}
                for recording in recordings:
                    run_files = sorted([
                        f for f in os.listdir(modality_path)
                        if f.endswith('_physio.tsv.gz') and task in f and recording in f
                    ])
                    run_names = [part for f in run_files for part in f.split('_') if part.startswith('run')]
                    runs[subject][session_dir][modality][task][recording] = run_names

# Get metadata
metadata = {}
for subject in subjects:
    metadata[subject] = {}
    for session_dir in sessions[subject]:
        metadata[subject][session_dir] = {}
        for modality in modalities:
            modality_path = os.path.join(bids_dir, subject, session_dir, modality)
            if not os.path.isdir(modality_path): continue
            metadata[subject][session_dir][modality] = {}
            for task in tasks:
                metadata[subject][session_dir][modality][task] = {}
                for recording in recordings:
                    metadata[subject][session_dir][modality][task][recording] = {}
                    json_files = [
                        f for f in os.listdir(modality_path)
                        if f.endswith('_physio.json') and task in f and recording in f
                    ]
                    for run in runs[subject][session_dir].get(modality, {}).get(task, {}).get(recording, []):
                        for json_file in json_files:
                            if f'{run}' in json_file:
                                with open(os.path.join(modality_path, json_file), 'r') as file:
                                    run_metadata = json.load(file)
                                metadata[subject][session_dir][modality][task][recording][run] = {
                                    'metadata': run_metadata,
                                    'filename': json_file
                                }

# Get filenames with full paths
filenames = {}
for subject in subjects:
    filenames[subject] = {}
    for session_dir in sessions[subject]:
        filenames[subject][session_dir] = {}
        for modality in modalities:
            modality_path = os.path.join(bids_dir, subject, session_dir, modality)
            if not os.path.isdir(modality_path): continue
            filenames[subject][session_dir][modality] = {}
            for task in tasks:
                filenames[subject][session_dir][modality][task] = {}
                for recording in recordings:
                    filenames[subject][session_dir][modality][task][recording] = {}
                    for run in runs[subject][session_dir].get(modality, {}).get(task, {}).get(recording, []):
                        filenames[subject][session_dir][modality][task][recording][run] = {}
                        for filetype in ['physioevents.json', 'physioevents.tsv.gz', 'physio.json', 'physio.tsv.gz']:
                            filenames[subject][session_dir][modality][task][recording][run][filetype] = [
                                os.path.join(modality_path, f) for f in os.listdir(modality_path)
                                if f.endswith(filetype) and task in f and recording in f and run in f
                            ]

## Preprocess data

In [4]:
# TO MOVE IN eyetrack_utils
def extract_data(tsvgz_fn, json_fn, x_coord_column, y_coord_column, pupil_column):
    """
    Load and process eye-tracking data and associated metadata from TSV and JSON files.
    Output only data in between start and stoptime as get in the JSON file.

    Args:
		tsvgz_fn (str): eye-tracking _physio.tsv.gz filename and path
	    json_fn (str): eye-tracking _physio.json filename and path
        x_coord_column (str): name of the X coordinate column
        y_coord_column (str): name of the X coordinate column
        pupil_column (str): name of the pupil column
  
    Returns:
        et_data_rec: eye-tracking dataframe of the data after the start signal
        et_metadata: eye-tracking dictionary of the metadata
    """

    import json 
    import pandas as pd
    
    # Extract metadata from the JSON
    with open(json_fn, 'r') as file:
        et_metadata = json.load(file)
    
    column_names = et_metadata['Columns']
    start_time = int(et_metadata['StartTime'][0])
    stop_time = int(et_metadata['StopTime'][0])

    # Add data columns
    et_metadata['x_coord_column'] = x_coord_column
    et_metadata['y_coord_column'] = y_coord_column
    et_metadata['pupil_column'] = pupil_column

    # Get data
    df = pd.read_csv(tsvgz_fn, 
                     compression='gzip', 
                     delimiter='\t', 
                     header=None,
                     names=column_names,  # Use the column names from JSON
                     na_values='n/a' # Treat 'n/a' as NaN
                    )
    
    # Get experiment data 
    # first column is necessarily the timestamps in physio tsv.gz
    et_data_rec = df[(df[df.columns[0]] >= start_time) & (df[df.columns[0]] <= stop_time)]
    
    return et_data_rec, et_metadata

def blink_removal(et_data, et_metadata, blink_removal_method, blink_buffer=None):
    """
    Replace blinks in eye-tracking data with NaN

    Args:
        et_data: eye-tracking dataframe
        et_metadata: eye-tracking dictionary of the metadata
        blink_removal_method: blink removal method
        blink_buffer: Period before and after detected blink to nan (default = None) 
        
    Returns:
        et_data_blinkless: dataframe cleaned from blink
    """
    import numpy as np

    if blink_removal_method == 'pupil_off':
        blink_bool = (et_data[et_metadata['pupil_column']] == 0)
        et_data_blinkless = et_data.copy()
        et_data_blinkless.loc[blink_bool, [et_metadata['x_coord_column'],
                                            et_metadata['y_coord_column'], 
                                            et_metadata['pupil_column']]] = np.nan
        
        # Add blink buffer
        if blink_buffer is not None:
            blink_buffer_samples = int(blink_buffer * et_metadata['SamplingFrequency'])
            blink_indices = et_data_blinkless.index[blink_bool]

            for blink_index in blink_indices:
                start_index = max(blink_index - blink_buffer_samples, et_data_blinkless.index.min())
                end_index = min(blink_index + blink_buffer_samples, et_data_blinkless.index.max())
                et_data_blinkless.loc[start_index:end_index, [et_metadata['x_coord_column'],
                                                               et_metadata['y_coord_column'], 
                                                               et_metadata['pupil_column']]] = np.nan

    return et_data_blinkless

def nan_removal(et_data, et_metadata):
    """
    Remove nan with either interpolation or extrapolation of the data.
    
    Args:
        et_data: eye-tracking dataframe
        et_metadata: eye-tracking dictionary of the metadata
    
    Returns:
        et_data_nanless: eye-tracking dataframe cleaned from nans
    """
    import numpy as np
    from scipy.interpolate import interp1d

    data_columns = [et_metadata['x_coord_column'], 
                    et_metadata['y_coord_column'], 
                    et_metadata['pupil_column']]
    
    # Create a copy of the original DataFrame to store the results
    et_data_nanless = et_data.copy()
    
    for data_column in data_columns:
        # Access the specific column directly from the DataFrame
        et_data_col = et_data_nanless[data_column].copy()
        
        # Create a mask for NaN values
        nan_indices = np.isnan(et_data_col)

        # Get the indices of the valid (non-NaN) values
        valid_indices = np.arange(len(et_data_col))[~nan_indices]
        valid_values = et_data_col[~nan_indices]
        
        # Create an interpolation function
        interp_func = interp1d(x=valid_indices,
                               y=valid_values, 
                               kind='linear',
                               bounds_error=False, 
                               fill_value="extrapolate")
        
        # Interpolate the NaN values
        et_data_col[nan_indices] = interp_func(np.arange(len(et_data_col))[nan_indices])
        
        # Update the original DataFrame with the interpolated values
        et_data_nanless[data_column] = et_data_col

    return et_data_nanless


In [5]:
# Imports
# from eyetrack_utils import *

for subject in subjects:
    for session in sessions[subject]:
        for modality in modalities:
            for task in tasks:
                for recording in recordings:
                    for run in runs[subject][session][modality][task][recording]:
                        print(f'Data: {subject} {session} {modality} {task} {run} {recording}')
                        
                        # Extract recording data (_rec)
                        print('\tLoading data/metadata...')
                        tsvgz_filename = filenames[subject][session][modality][task][recording][run]['physio.tsv.gz']
                        json_filename = filenames[subject][session][modality][task][recording][run]['physio.json']
                        data_rec, metadata_rec = extract_data(tsvgz_fn=tsvgz_filename[0],
                                                              json_fn=json_filename[0],
                                                              x_coord_column=x_coord_column,
                                                              y_coord_column=y_coord_column,
                                                              pupil_column=pupil_column
                                                             )

                        # Remove blinks (_blinkless)
                        print('\tRemoving blinks...')
                        data_rec_blinkless = blink_removal(et_data=data_rec,
                                                           et_metadata=metadata_rec,
                                                           blink_removal_method=blink_removal_method,
                                                           blink_buffer=blink_buffer)

                        # Remove nans with inter- and extrapolation of missing data (_nanless)
                        print('\tRemoving nans...')
                        data_rec_blinkless_nanless = nan_removal(et_data=data_rec_blinkless,
                                                                 et_metadata=metadata_rec)
                        


                        # Convert to degrees of visual angle
                        # Center gaze position on screen center
                        # Detrending gaze position
                        # Normalize pupil data
                        # Downsample gaze and pupil
                        # Smoothing gaze and pupil
                        
                        # Save the preprocessed data

Data: sub-01 ses-01 func task-pMF run-01 recording-eye1
	Loading data/metadata...
	Removing blinks...
	Removing nans...


NameError: name 'fff' is not defined

In [ ]:
data_rec_blinkless

In [ ]:
data_rec_blinkless_nanless